# 03 — Leakage-safe feature engineering
Create historical inputs and exact-date future targets, then manually verify one row.

In [ ]:
%pip install -q -e .
import pandas as pd
from agridecision.features.tabular import build_supervised_frame, select_model_features

In [ ]:
prices = pd.read_csv('data/processed/mandi_prices.csv', parse_dates=['arrival_date'])
supervised = build_supervised_frame(
    prices, horizon_days=7, shock_threshold=0.15,
    lags=(1, 7, 14, 28), rolling_windows=(7, 14, 28)
)
print(supervised.shape)
supervised.head()

In [ ]:
categorical, numeric = select_model_features(supervised)
print('Categorical features:', categorical)
print('Numeric features:', numeric)
assert 'target_modal_price' not in categorical + numeric
assert 'target_return' not in categorical + numeric

## Manual leakage check
For one sample row, compare the stored target with the same market exactly seven calendar days later and the lag with one day earlier.

In [ ]:
sample = supervised.iloc[len(supervised) // 2]
same_series = prices[(prices['market'] == sample['market']) & (prices['commodity'] == sample['commodity']) & (prices['variety'] == sample['variety'])]
future = same_series.loc[same_series['arrival_date'] == sample['arrival_date'] + pd.Timedelta(days=7), 'modal_price']
previous = same_series.loc[same_series['arrival_date'] == sample['arrival_date'] - pd.Timedelta(days=1), 'modal_price']
print('Stored target:', sample['target_modal_price'], 'Manual target:', future.iloc[0])
print('Stored lag 1:', sample['price_lag_1'], 'Manual lag:', previous.iloc[0])

In [ ]:
supervised.to_csv('data/processed/supervised_features.csv', index=False)
print('Saved modelling table.')